# Phase 4 — YOLOv8 Implementation

Replaces only the detector (YOLOv8n via Ultralytics), keeping dataset, augmentation, and optimizer equivalent to the Faster R-CNN baseline (Phase 3).

**Methodological notes:**
- Ultralytics' default low-probability (p=0.01) Albumentations transforms (Blur, MedianBlur, ToGray, CLAHE) were disabled by uninstalling the `albumentations` package, since they have no equivalent in the Faster R-CNN pipeline.
- Mosaic, mixup, scale, translate, shear, and perspective augmentations (Ultralytics defaults) were disabled; only horizontal flip, rotation (±10°), and brightness/contrast jitter were kept, matching Faster R-CNN.
- Same optimizer (SGD, lr=0.001, momentum=0.9, weight_decay=0.0005), same batch size (2), and same early-stopping patience (15) as Faster R-CNN.

In [ ]:
!pip install ultralytics -q
!pip uninstall albumentations -y -q

In [ ]:
import yaml

data_yaml = {
    "path": "/content/brain-tumor-detection-comparison/data/yolo",
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "names": {0: "Glioma", 1: "Meningioma", 2: "Pituitary", 3: "No Tumor"},
}
with open("data_yolo.yaml", "w") as f:
    yaml.dump(data_yaml, f)

In [ ]:
from ultralytics import YOLO

model_yolo = YOLO("yolov8n.pt")

results = model_yolo.train(
    data="data_yolo.yaml",
    epochs=100,
    patience=15,
    batch=2,
    imgsz=640,
    optimizer="SGD",
    lr0=0.001,
    momentum=0.9,
    weight_decay=0.0005,
    fliplr=0.5,
    flipud=0.0,
    degrees=10,
    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.2,
    mosaic=0.0,
    mixup=0.0,
    scale=0.0,
    translate=0.0,
    shear=0.0,
    perspective=0.0,
    project="/content/drive/MyDrive/brain-tumor-project/yolo_run",
    name="yolo_run_v2",
    seed=42,
)

## Results

Training stopped early at epoch 53 (patience=15); best model at **epoch 38**.

![Training curves](../results/figures/yolo_training_curves.png)

| Model | Best Epoch | Precision | Recall | mAP@0.5 | mAP@0.5:0.95 | Params (M) | GFLOPs | FPS | Model size (MB) |
|---|---|---|---|---|---|---|---|---|---|
| Faster R-CNN | 14 | 0.8366 | 0.9526 | 0.9527 | 0.6711 | 43.27 | 280.81 | 7.15 | 329.69 |
| YOLOv8n | 38 | 0.9570 | 0.9410 | 0.9790 | 0.7350 | 3.01 | 8.10 | 181.82 | 6.20 |

Full table: `results/tables/model_comparison.csv`